# Agentic AutoML — Experiments

Run the agentic pipeline on any tabular dataset by setting `DATA_PATH` and `TARGET`.

**Prerequisites:** add your Anthropic API key to the `.env` file in the project root.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on the Python path regardless of how the notebook is launched
PROJECT_ROOT = Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()  # loads ANTHROPIC_API_KEY from .env

print(f"Project root : {PROJECT_ROOT}")
print(f"SQLite store : {PROJECT_ROOT / 'storage' / 'runs.db'}")
print(f"ChromaDB dir : {PROJECT_ROOT / 'storage' / 'chroma_db'}")

In [ ]:
from src.data.loader import load_data, split_data
from src.agents.profiler import generate_profile
from src.agents.issue_detector import detect_issues
from src.orchestrator import run_agentic_pipeline

## 1. Load dataset

In [ ]:
DATA_PATH = "data/titanic.csv"   # ← change to your dataset
TARGET    = "Survived"           # ← change to your target column

df = load_data(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 2. Profile & inspect detected issues

In [ ]:
profile = generate_profile(df, TARGET)
issues  = detect_issues(profile, df)

print(f"Task type : {profile.target_type.value}")
print(f"Duplicates: {profile.n_duplicates}")
if profile.class_distribution:
    print(f"Classes   : {profile.class_distribution}")

print(f"\nDetected issues ({len(issues)}):")
for issue in issues:
    print(f"  [{issue.severity.value.upper():<6}] {issue.issue_type.value:<22}  {issue.description}")

## 3. Run the agentic pipeline

Results are automatically persisted to `storage/runs.db` (SQLite) and `storage/chroma_db` (ChromaDB).
Both paths are fixed relative to the project root — they do not depend on the working directory.

In [ ]:
result = run_agentic_pipeline(
    df,
    target            = TARGET,
    max_rounds        = 3,
    n_plans_per_round = 3,
    cv                = 5,
    score_threshold   = 0.90,
    verbose           = True,
    # db_path and chroma_dir default to storage/ in the project root
)

## 4. Results

In [ ]:
print(f"Run ID      : {result.run_id}")
print(f"Iterations  : {result.n_iterations}")
print(f"Converged   : {result.converged}")
print(f"Best score  : {result.best_result.score:.4f}")
print(f"CV std      : {result.best_result.cv_std:.4f}")
print(f"Metrics     : {result.best_result.metric_values}")
print(f"\nBest plan:")
print(f"  model              = {result.best_plan.model}")
print(f"  imputation         = {result.best_plan.imputation}")
print(f"  encoding           = {result.best_plan.encoding}")
print(f"  scaling            = {result.best_plan.scaling}")
print(f"  outlier_handling   = {result.best_plan.outlier_handling}")
print(f"  imbalance_strategy = {result.best_plan.imbalance_strategy}")

## 5. Iteration history

In [ ]:
import pandas as pd

rows = []
for rec in result.history:
    rows.append({
        "iteration"  : rec.iteration,
        "model"      : rec.plan.model,
        "imputation" : rec.plan.imputation,
        "encoding"   : rec.plan.encoding,
        "scaling"    : rec.plan.scaling,
        "score"      : round(rec.result.score, 4),
        "cv_std"     : round(rec.result.cv_std, 4),
        **{k: round(v, 4) for k, v in rec.result.metric_values.items() if isinstance(v, float)},
    })

pd.DataFrame(rows).sort_values("score", ascending=False)

## 6. Use the best pipeline for prediction

In [ ]:
X_train, X_test, y_train, y_test = split_data(df, TARGET)
y_pred = result.best_pipeline.predict(X_test)

print(f"Test predictions (first 10): {y_pred[:10]}")

## 7. Inspect past runs (SQLite)

In [ ]:
from src.memory.run_store import RunStore
from src.orchestrator import _DEFAULT_DB_PATH

with RunStore(_DEFAULT_DB_PATH) as store:
    print("All run IDs stored so far:")
    for run_id in store.list_runs():
        attempts = store.load_attempts(run_id)
        best = max(attempts, key=lambda r: r.result.score)
        print(f"  {run_id[:8]}...  {len(attempts)} attempts  best_score={best.result.score:.4f}")

---
## 8. Ablation Study

Runs four variants of the system on the same dataset and compares them side-by-side.

| Variant | Memory | Feedback loop |
|---------|--------|---------------|
| Rule-based baseline | — | — |
| Search-based baseline (Optuna) | — | — |
| Agentic — no memory | ✗ | ✓ |
| Agentic — no feedback (1 round) | ✓ | ✗ |
| **Full agentic system** | **✓** | **✓** |

In [ ]:
from src.baselines.rule_based import run_rule_based
from src.baselines.search_based import run_search_based

ABLATION_TRIALS = 50   # Optuna trials for search-based baseline
ABLATION_CV     = 5    # must match agentic system
ABLATION_ROUNDS = 3    # max rounds for agentic variants

In [ ]:
# --- Variant 1: Rule-based baseline ---
rb = run_rule_based(df, TARGET, cv=ABLATION_CV)
print(f"Rule-based  score={rb.score:.4f}  metrics={rb.metric_values}  runtime={rb.runtime_secs:.1f}s")

In [ ]:
# --- Variant 2: Search-based baseline (Optuna TPE) ---
sb = run_search_based(df, TARGET, n_trials=ABLATION_TRIALS, cv=ABLATION_CV)
print(f"Search-based  score={sb.score:.4f}  metrics={sb.metric_values}  runtime={sb.runtime_secs:.1f}s")

In [ ]:
# --- Variant 3: Agentic system without cross-run memory ---
ag_no_mem = run_agentic_pipeline(
    df,
    target            = TARGET,
    max_rounds        = ABLATION_ROUNDS,
    n_plans_per_round = 3,
    cv                = ABLATION_CV,
    score_threshold   = 0.90,
    use_memory        = False,   # ablation: disable ChromaDB
    verbose           = True,
)
print(f"Agentic (no memory)  score={ag_no_mem.best_result.score:.4f}  "
      f"iterations={ag_no_mem.n_iterations}  metrics={ag_no_mem.best_result.metric_values}")

In [ ]:
# --- Variant 4: Agentic system without feedback loop (single round) ---
ag_no_fb = run_agentic_pipeline(
    df,
    target            = TARGET,
    max_rounds        = 1,       # ablation: only one Planner call, no iterative refinement
    n_plans_per_round = 3,
    cv                = ABLATION_CV,
    score_threshold   = 0.90,
    use_memory        = True,
    verbose           = True,
)
print(f"Agentic (no feedback)  score={ag_no_fb.best_result.score:.4f}  "
      f"iterations={ag_no_fb.n_iterations}  metrics={ag_no_fb.best_result.metric_values}")

In [ ]:
# --- Variant 5: Warm-start experiment ---
# Run the full system twice on the same dataset. The second run retrieves the
# best plan from the first run via ChromaDB and uses it to warm-start the Planner.
# A lower iteration count or higher score on the second run indicates memory benefit.
ag_warm1 = run_agentic_pipeline(
    df, target=TARGET, max_rounds=ABLATION_ROUNDS, n_plans_per_round=3,
    cv=ABLATION_CV, score_threshold=0.90, use_memory=True, verbose=False,
)
ag_warm2 = run_agentic_pipeline(
    df, target=TARGET, max_rounds=ABLATION_ROUNDS, n_plans_per_round=3,
    cv=ABLATION_CV, score_threshold=0.90, use_memory=True, verbose=False,
)
print(f"Warm-start run 1  score={ag_warm1.best_result.score:.4f}  iterations={ag_warm1.n_iterations}")
print(f"Warm-start run 2  score={ag_warm2.best_result.score:.4f}  iterations={ag_warm2.n_iterations}")
print(f"Δ score = {ag_warm2.best_result.score - ag_warm1.best_result.score:+.4f}  "
      f"Δ iterations = {ag_warm2.n_iterations - ag_warm1.n_iterations:+d}")

In [ ]:
# --- Summary table ---
import pandas as pd

primary_metric = next(iter(result.best_result.metric_values))

summary = pd.DataFrame([
    {"variant": "Rule-based",         "score": rb.score,
     "primary_metric": rb.metric_values.get(primary_metric),
     "cv_std": rb.cv_std, "configs": rb.n_configs_evaluated, "iterations": "—"},
    {"variant": "Search-based (Optuna)", "score": sb.score,
     "primary_metric": sb.metric_values.get(primary_metric),
     "cv_std": sb.cv_std, "configs": sb.n_configs_evaluated, "iterations": "—"},
    {"variant": "Agentic — no memory", "score": ag_no_mem.best_result.score,
     "primary_metric": ag_no_mem.best_result.metric_values.get(primary_metric),
     "cv_std": ag_no_mem.best_result.cv_std, "configs": "—",
     "iterations": ag_no_mem.n_iterations},
    {"variant": "Agentic — no feedback", "score": ag_no_fb.best_result.score,
     "primary_metric": ag_no_fb.best_result.metric_values.get(primary_metric),
     "cv_std": ag_no_fb.best_result.cv_std, "configs": "—",
     "iterations": ag_no_fb.n_iterations},
    {"variant": "Full agentic system",  "score": result.best_result.score,
     "primary_metric": result.best_result.metric_values.get(primary_metric),
     "cv_std": result.best_result.cv_std, "configs": "—",
     "iterations": result.n_iterations},
])
summary = summary.rename(columns={"primary_metric": primary_metric})
summary.sort_values("score", ascending=False).reset_index(drop=True)